In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005


config = Config()

In [4]:
print(utils.recall_vals_one_row([1, 2, 3], 2))
print(utils.recall_vals_one_row([1, 2, 3], 4))

True
False


In [5]:
utils.recall_several_targets(candidates=[[1, 2, 3], [1]], targets=[[1, 3, 2], [1, 2]])

0.8

In [6]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=16, progress_bar=True, use_memory_fs=False)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [7]:
%%time

PATH_DATA = '../data/02_intermediate/'
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
df_train = df_train.to_pandas()
df_test = df_test.to_pandas()
# df_test = catalog.load("df_test")

df_train.name = "df_train"
df_test.name = "df_test"

df_train = utils.make_date_stats(df_train)
df_test = utils.make_date_stats(df_test)

df_train.drop(['date_day'], axis=1, inplace=True)
df_test.drop(['date_day'], axis=1, inplace=True)

df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

CPU times: user 1min 53s, sys: 20 s, total: 2min 13s
Wall time: 1min 56s


In [8]:
%%time

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

CPU times: user 6min 58s, sys: 1min 1s, total: 8min
Wall time: 1min 4s


In [9]:
del df_train, df_test
gc.collect()

2624

In [10]:
# ### IF DEBUG PIPELINE !!!

# df = df.filter(pl.col("user") % 100 == 0)

In [11]:
counts_aid = df.groupby([config.col_aid]).count().sort("count", reverse=True)

In [12]:
df.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,num_action
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,i32
14571581,1100210,1662328791563,"""clicks""",0,2022-09-04 23:59:51.563000064,4,35,6,23,0
14571515,781377,1662328789887,"""clicks""",0,2022-09-04 23:59:49.887000064,4,35,6,23,0
14571170,336732,1662328788088,"""clicks""",0,2022-09-04 23:59:48.088,4,35,6,23,0
14568399,104775,1662328781349,"""clicks""",0,2022-09-04 23:59:41.348999936,4,35,6,23,0
14571580,202353,1662328781067,"""clicks""",0,2022-09-04 23:59:41.067000064,4,35,6,23,0


In [13]:
counts_aid.head()

aid,count
i64,u32
1460571,137874
485256,135892
108125,124885
29735,116215
1733943,106512


In [14]:
df.shape

(223644219, 11)

In [15]:
df = df.to_pandas()

In [16]:
def split_data(df, validation_cut, test_users):
    df_val = df[
        (df[config.col_num_action] <= validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_val_seq = df[
        (df[config.col_num_action] > validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_train = df.drop(df_val.index)
    return df_train, df_val, df_val_seq

In [17]:
%%time

X_train, X_val, X_val_seq = split_data(df, validation_cut=2, test_users=test_users)

CPU times: user 26.9 s, sys: 3.33 s, total: 30.2 s
Wall time: 30.3 s


In [18]:
%%time

trs = 20
counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
pad = len(counts_aid_filtered)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

CPU times: user 611 ms, sys: 25.7 ms, total: 637 ms
Wall time: 631 ms


OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=954978)

In [19]:
%%time

X_train[config.col_aid] = ordinal_encoder.transform(
    X_train[config.col_aid].values.reshape(-1, 1)
).astype(int)
X_val_seq[config.col_aid] = ordinal_encoder.transform(
    X_val_seq[config.col_aid].values.reshape(-1, 1)
).astype(int)

CPU times: user 1min 31s, sys: 3.31 s, total: 1min 34s
Wall time: 1min 35s


In [20]:
%%time

X_train = pl.from_pandas(X_train)
X_train = X_train.sort(config.col_ts)

X_val = pl.from_pandas(X_val)
X_val = X_val.sort(config.col_ts)

X_val_seq = pl.from_pandas(X_val_seq)
X_val_seq = X_val_seq.sort(config.col_ts)

CPU times: user 2min 55s, sys: 50 s, total: 3min 45s
Wall time: 19.3 s


In [21]:
X_train.filter(pl.col(config.col_user) == 0)

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour,num_action
i64,i64,i64,str,i32,datetime[ns],i8,i8,i8,i8,i32
0,780767,1659304800025,"""clicks""",0,2022-08-01 00:00:00.024999936,1,31,0,0,275
0,804697,1659304904511,"""clicks""",0,2022-08-01 00:01:44.511000064,1,31,0,0,274
0,673857,1659367439426,"""clicks""",1,2022-08-01 17:23:59.425999872,1,31,0,17,273
0,8442,1659367719997,"""clicks""",1,2022-08-01 17:28:39.996999936,1,31,0,17,272
0,917035,1659367871344,"""clicks""",1,2022-08-01 17:31:11.344,1,31,0,17,271
0,593124,1659367885796,"""clicks""",1,2022-08-01 17:31:25.796,1,31,0,17,270
0,849118,1659369893840,"""carts""",1,2022-08-01 18:04:53.840,1,31,0,18,269
0,237160,1659369898050,"""carts""",1,2022-08-01 18:04:58.049999872,1,31,0,18,268
0,156985,1659370027105,"""orders""",1,2022-08-01 18:07:07.104999936,1,31,0,18,266


In [22]:
%%time

X_train = (
    X_train.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
)

X_val = (
    X_val.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

X_val_seq = (
    X_val_seq.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

CPU times: user 4min 32s, sys: 37.2 s, total: 5min 9s
Wall time: 1min 7s


In [23]:
X_train.shape

(13265267, 2)

In [24]:
X_train.filter(pl.col(config.col_user) == 0)

user,aid
i64,list[i64]
0,"[780767, 804697, ... 83448]"


In [25]:
# dict_map_targets = dict()
# dict_map_targets["clicks"] = 0
# dict_map_targets["carts"] = 1
# dict_map_targets["orders"] = 2

In [26]:
counts_aid["aid"].min(), counts_aid["aid"].max()

(0, 1855602)

In [27]:
pad

954978

In [28]:
# %%time

# df_train = df_train.to_pandas()
# df_train['aid_filtered'] = ordinal_encoder.transform(df_train['aid'].values.reshape(-1, 1)).astype(int)
# df_train = pl.from_pandas(df_train)

In [29]:
# counts_aid_type.filter((pl.col("type") == "orders") & (pl.col("count") >= 1)).select(
#     ["aid"]
# ).unique().shape

In [30]:
# counts_aid_type

In [31]:
counts_aid.tail()

aid,count
i64,u32
1089137,3
967977,3
1582417,3
1054721,3
1166769,3


In [32]:
counts_aid.filter(pl.col("aid") == 1)

aid,count
i64,u32
1,34


In [33]:
# len(counts_aid), len(counts_aid_filtered)

In [34]:
# %%time

# token_to_idx = dict()
# # token_to_idx[-1] = 0 # pad
# token_to_idx[0] = 0 # unknown / pad

# idx = 1
# for i in tqdm(range(len(counts_aid["aid"]))):
#     aid = counts_aid["aid"][i]
#     cnt = counts_aid["count"][i]
#     if cnt > 4:
#         token_to_idx[aid] = idx
#         idx += 1
#     # else:
#     #     token_to_idx[aid] = 0

# idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [35]:
# len(token_to_idx)

In [36]:
# min(list(token_to_idx.values()))

In [37]:
# token_to_idx[0]

In [38]:
# max_len = 10
# # pad = counts_aid["aid"].max() + 1

# # def get_id_of_token(token):
# #     if token in set(list(token_to_idx.keys())[1:]):
# #         idx = token_to_idx[token]
# #     else:
# #         idx = 0  # unknown
# #     return idx


# # def get_token_if_id(idx):
# #     if idx in set(list(idx_to_token.keys())[1:]):
# #         token = idx_to_token[idx]
# #     else:
# #         token = 0  # unknown
# #     return token


# def to_matrix(names, max_len=max_len, pad=pad, dtype="int32", batch_first=True):
#     """Casts a list of names into rnn-digestible matrix"""

#     max_len = max_len or max(map(len, names))
#     names_ix = np.zeros([len(names), max_len], dtype) + pad

#     for i in tqdm(range(len(names))):
#         cur_name = names[i]
#         line_ix = cur_name[-max_len:]  # last max_len elements
#         # cur_name = cur_name[-max_len:]  # last max_len elements
#         # line_ix = [
#         #     get_id_of_token(cur_name[c]) for c in range(len(cur_name)) if c < max_len
#         # ]
#         names_ix[i, -len(line_ix) :] = line_ix

#     if not batch_first:  # convert [batch, time] into [time, batch]
#         names_ix = np.transpose(names_ix)

#     return names_ix

In [39]:
l = [1, 2, 3]
print(utils.split_list_randomly(l=l, seed=1))
print(utils.split_list_randomly(l=l, seed=2))

([1, 2], [3])
([1], [2, 3])


In [40]:
X_train.head()

user,aid
i64,list[i64]
3386624,"[901386, 831662, ... 621802]"
3604608,"[468967, 495302, ... 887423]"
1606144,"[36655, 262442, ... 883046]"
5957504,"[51935, 545858, ... 545858]"
3732352,"[564728, 234600, ... 385989]"


In [41]:
X_train.shape

(13265267, 2)

In [42]:
X_val.head()

user,aid
i64,list[i64]
14056384,[638132]
12976128,"[247770, 247770]"
13783488,[1643394]
13987328,"[1832326, 1686992]"
14144000,[1124416]


In [43]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: x[:-1])
X_train[config.col_target] = X_train[config.col_aid].parallel_apply(lambda x: x[-1])
X_train = pl.from_pandas(X_train)

CPU times: user 2min 19s, sys: 28.2 s, total: 2min 47s
Wall time: 2min 54s


In [44]:
X_train.head()

user,aid,train_seq,target
i64,list[i64],list[i64],i64
3386624,"[901386, 831662, ... 621802]","[901386, 831662, ... 34008]",621802
3604608,"[468967, 495302, ... 887423]","[468967, 495302, ... 954978]",887423
1606144,"[36655, 262442, ... 883046]","[36655, 262442, ... 227627]",883046
5957504,"[51935, 545858, ... 545858]","[51935, 545858, 46666]",545858
3732352,"[564728, 234600, ... 385989]","[564728, 234600, ... 420255]",385989


In [45]:
# %%time

# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[:-1]).alias(config.col_train_seq)))
# X_train = X_train.with_column((pl.col(config.col_aid).apply(lambda x: x[-1]).alias(config.col_target)))

In [46]:
# %%time

# X_train['val'] = X_train["val"].progress_apply(lambda x: x[0]) #orig_vals

# col_aids = 'train'
# X_train = pl.from_pandas(X_train)
# X_train = (
#     X_train.groupby(["user", 'val'])
#     .agg(
#         [
#             pl.col(col_aids).list(),
#         ]
#     )
#     # .filter(pl.col(col_aids).apply(lambda x: len(x)) >= 2)
# ).sort('user').to_pandas()

# X_train = X_train[['user', 'train', 'val']]

In [47]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
886080,31
679488,2
939648,31
99072,7
641152,17
9472,1
768448,1
905280,2
625984,2


In [48]:
%%time

X_train = X_train.to_pandas()

max_len = config.max_len

list_train = list()
list_target = list()
list_user = list()

n_iter = 5
for i in tqdm(range(len(X_train))):
    row = X_train.iloc[i]
    row_train = list(row[config.col_train_seq])
    target_cur_main = row[config.col_target]
    len_row_train = len(row_train)
    if len_row_train == 1 and target_cur_main < pad:
        list_train.append(tuple(row_train))
        list_target.append(target_cur_main)
        list_user.append(row[config.col_user])
    else:
        n_iter_cur = min(n_iter, len_row_train - 1)
        train_cur_main = utils.truncate_list(l=row_train, max_len=max_len, from_start=False)
        if target_cur_main < pad:
            list_train.append(tuple(train_cur_main))
            list_target.append(target_cur_main)
            list_user.append(row[config.col_user])

        for seed in range(n_iter_cur):
            train_cur, target_cur = utils.split_list_randomly(l=row_train, seed=seed)
            target_cur = target_cur[0]
            if target_cur < pad:
                train_cur = utils.truncate_list(l=train_cur, max_len=max_len, from_start=False)
                list_train.append(tuple(train_cur))
                list_target.append(target_cur)
                list_user.append(row[config.col_user])

X_train = pd.DataFrame()
X_train[config.col_user] = list_user
X_train[config.col_train_seq] = list_train
X_train[config.col_target] = list_target
X_train.drop_duplicates(inplace=True)
X_train.reset_index(drop=True, inplace=True)

X_train = pl.from_pandas(X_train)

  0%|          | 0/13265267 [00:00<?, ?it/s]

CPU times: user 39min 3s, sys: 49.6 s, total: 39min 52s
Wall time: 39min 45s


In [49]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False))
# X_train = pl.from_pandas(X_train)

In [50]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
3386624,"[831662, 433725, ... 34008]",621802
3386624,"[671720, 654957, ... 34008]",433725
3386624,"[901386, 831662, ... 831662]",433725
3386624,"[901386, 831662, ... 433725]",433725
3386624,"[831662, 654957, ... 433725]",34008


In [51]:
X_train = X_train.filter(pl.col(config.col_target) != pad)

In [52]:
X_train[config.col_target].max(), pad  # (31207, 31211)

(954977, 954978)

In [53]:
X_train.shape

(45780769, 3)

In [54]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
946688,45
461952,32
573184,4
60224,15
905984,8
947904,7
209088,2
839104,23
296256,6


In [55]:
X_val_seq.head()

user,aid
i64,list[i64]
13241664,"[946628, 954978, ... 330598]"
13567936,"[575808, 575808, ... 424251]"
13345984,"[801573, 801573]"
13573184,"[166298, 350100, ... 501755]"
13589952,"[471462, 469961]"


In [56]:
X_val_seq = X_val_seq.to_pandas()
X_val_seq[config.col_train_seq] = X_val_seq[config.col_aid].parallel_apply(
    lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
)
X_val_seq = pl.from_pandas(X_val_seq)

In [57]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13241664,"[946628, 954978, ... 330598]","[946628, 954978, ... 330598]"
13567936,"[575808, 575808, ... 424251]","[575808, 575808, ... 424251]"
13345984,"[801573, 801573]","[801573, 801573]"
13573184,"[166298, 350100, ... 501755]","[954978, 954978, ... 501755]"
13589952,"[471462, 469961]","[471462, 469961]"


In [58]:
X_val = X_val.join(
    X_val_seq[[config.col_user, config.col_train_seq]], how="inner", on=config.col_user
)

In [59]:
X_val = X_val.rename({config.col_aid: config.col_target})

In [60]:
X_val.filter(pl.col(config.col_user) == 13833088)

user,target,train_seq
i64,list[i64],list[i64]
13833088,"[998481, 998481, 226025]","[261470, 300201, ... 513695]"


In [61]:
pad

954978

In [62]:
# df_train[col_aids].value_counts().sort("counts", reverse=True)

In [63]:
# df_train_clicks.drop([col_aids, "split"], axis=1, inplace=True)

In [64]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [pl.col("aid").apply(lambda x: utils.split_list_randomly(list(x))).alias("split")]
# )

In [65]:
# %%time

# df_train_clicks = df_train_clicks.with_columns(
#     [
#         pl.col("split").apply(lambda x: x[0]).alias("train"),
#         pl.col("split").apply(lambda x: x[1]).alias("val"),
#     ]
# )

# for col in ["aid", "split"]:
#     df_train_clicks.drop_in_place(col)

# df_train_clicks = df_train_clicks.sort("user")

In [66]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.to_pandas()

In [67]:
# %%time

# df_train_clicks_mapped["val"] = df_train_clicks_mapped["val"].parallel_apply(
#     lambda x: get_id_of_token(x[0])
# )

In [68]:
# shape_orig = df_train_clicks_mapped.shape
# df_train_clicks_mapped = df_train_clicks_mapped[
#     df_train_clicks_mapped["val"] != 0
# ].reset_index(drop=True)
# print(shape_orig, "->", df_train_clicks_mapped.shape)

In [69]:
# (12773, 4)

In [70]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
3386624,"[831662, 433725, ... 34008]",621802
3386624,"[671720, 654957, ... 34008]",433725
3386624,"[901386, 831662, ... 831662]",433725
3386624,"[901386, 831662, ... 433725]",433725
3386624,"[831662, 654957, ... 433725]",34008


In [71]:
X_val.head()

user,target,train_seq
i64,list[i64],list[i64]
14317248,"[1257293, 1257293, 1257293]","[63656, 404040, ... 647060]"
13172288,"[644778, 1674216, 644778]","[799059, 954978]"
13225728,"[270070, 943295, 943295]",[948806]
13713600,"[669989, 14530, 1006198]",[29121]
13826560,"[356824, 356824, 356824]","[633595, 624605, ... 365077]"


In [72]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].progress_apply(lambda x: utils.truncate_list(list(x), max_len=max_len))

In [73]:
# %%time

# df_train_clicks["train"] = list(to_matrix(df_train_clicks["train"]))

In [74]:
# df_train_clicks["val"].value_counts()

In [75]:
# df_train_clicks.head()

In [76]:
# df_train_clicks["train"][0]

In [77]:
# %%time

# df_train_clicks["train"] = df_train_clicks["train"].parallel_apply(
#     lambda x: to_matrix([x])[0]
# )

In [78]:
# %%time

# df_train_clicks_mapped = df_train_clicks.clone()
# df_train_clicks_mapped = df_train_clicks_mapped.with_columns(
#     [
#         pl.col("train").apply(lambda x: to_matrix(list(x))[0]).alias("train"),
#         pl.col("val").apply(lambda x: to_matrix(list('val')[:1])[0]).alias("val"),
#     ]
# )
# df_train_clicks_mapped = df_train_clicks_mapped.sort("user")

In [79]:
# class Dataset_RNN(Dataset):
#     def __init__(self, df, mode):

#         self.df = df  # .reset_index(drop=True)
#         self.mode = mode

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, index):
#         row = self.df.iloc[index]
#         user = row["user"]
#         aids_train = row["train"]
#         # aids_train = to_matrix([aids_train])[0]
#         aids_val = row["val"]
#         # aids_val = to_matrix([aids_val])[0]

#         if self.mode == "test":
#             return torch.tensor(aids_train).long()
#         else:
#             return torch.tensor(aids_train).long(), torch.tensor(aids_val).float()

In [80]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [81]:
# min(600, round(1.6 * n_cat**0.56))

n_vocab = pad + 1
# hidden_dim = 128
# embedding_dim = min(600, round(1.6 * pad**0.56))
# embedding_dim = 100

In [82]:
# class Simple_LSTM(nn.Module):
#     def __init__(
#         self,
#         out_features_num,
#         n_vocab=n_vocab,
#         hidden_dim=hidden_dim,
#         embedding_dim=embedding_dim,
#         dropout=0.1,
#     ):
#         super(Simple_LSTM, self).__init__()
#         self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=True)
#         self.hidden_dim = hidden_dim
#         self.out_features_num = out_features_num

#         self.lstm_desc = nn.LSTM(
#             embedding_dim,
#             hidden_dim,
#             dropout=dropout,
#             num_layers=2,
#             bidirectional=True,
#             batch_first=True,
#         )

#         self.pooling_result_dimension = self.hidden_dim * 2
#         self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

#         self.attention_descs = Attention(self.hidden_dim * 2, batch_first=True)
#         self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
#         self.dropout1 = nn.Dropout(dropout, inplace=True)
#         self.fc1 = nn.Linear(self.dim_after_concat, self.pooling_result_dimension)
#         self.dropout2 = nn.Dropout(dropout, inplace=True)
#         self.fc2 = nn.Linear(self.pooling_result_dimension, self.out_features_num)

#         # self.category_embedding = self._create_embedding_projection(cardinality=10, embed_size=6, padding_idx=None, scale_grad_by_freq=False)
#         # self.subcategory_embedding = self._create_embedding_projection(cardinality=50, embed_size=14, padding_idx=None, scale_grad_by_freq=False)
#         # self.region_embedding = self._create_embedding_projection(cardinality=85, embed_size=19, padding_idx=None, scale_grad_by_freq=False)

#     def _create_embedding_projection(
#         self,
#         cardinality,
#         embed_size,
#         add_missing=True,
#         padding_idx=0,
#         scale_grad_by_freq=False,
#     ):
#         add_missing = add_missing * 1
#         return nn.Embedding(
#             num_embeddings=cardinality + add_missing,
#             embedding_dim=embed_size,
#             padding_idx=padding_idx,
#             scale_grad_by_freq=scale_grad_by_freq,
#         )

#     def forward(self, aids):
#         batch_size = aids.shape[0]

#         # category_embed = self.category_embedding(category)
#         # subcategory_embed = self.subcategory_embedding(subcategory)
#         # region_embed = self.region_embedding(region)

#         embedded_aids = self.embeddings(aids)

#         states_aids, hh_aids = self.lstm_desc(embedded_aids)

#         lengths_aids = [x_.size(1) for x_ in states_aids]
#         atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
#         atten_aids = atten_aids.view(batch_size, -1)

#         features = torch.cat(
#             [
#                 # subcategory_embed, category_embed, region_embed,
#                 atten_aids
#             ],
#             dim=-1,
#         )
#         # features = self.bn1(features)
#         features = self.dropout1(features)
#         features = features.view(features.size(0), -1)
#         features = self.fc1(features)
#         features = self.dropout2(features)
#         features = self.fc2(features)

#         return features

In [83]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        dropout=0.1,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab

        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens, targets=None):
        batch_size = aids.shape[0]

        embedded_aids = self.embeddings(aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.fc2(features)

        return features

In [84]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
3386624,"[831662, 433725, ... 34008]",621802
3386624,"[671720, 654957, ... 34008]",433725
3386624,"[901386, 831662, ... 831662]",433725
3386624,"[901386, 831662, ... 433725]",433725
3386624,"[831662, 654957, ... 433725]",34008


In [85]:
# X_val["val"].value_counts()

In [86]:
X_val.head()

user,target,train_seq
i64,list[i64],list[i64]
14317248,"[1257293, 1257293, 1257293]","[63656, 404040, ... 647060]"
13172288,"[644778, 1674216, 644778]","[799059, 954978]"
13225728,"[270070, 943295, 943295]",[948806]
13713600,"[669989, 14530, 1006198]",[29121]
13826560,"[356824, 356824, 356824]","[633595, 624605, ... 365077]"


In [87]:
X_train.shape, X_val.shape

((45780769, 3), (466825, 3))

In [88]:
# TRAIN_BATCH_SIZE = 128
# INFER_BATCH_SIZE = 128

# dataset_train = Dataset_RNN(X_train, mode="train")
# dataset_local_valid = Dataset_RNN(X_val, mode="train")

# train_loader = torch.utils.data.DataLoader(
#     dataset_train,
#     batch_size=TRAIN_BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,
#     drop_last=False,
# )
# local_valid_loader = torch.utils.data.DataLoader(
#     dataset_local_valid,
#     batch_size=INFER_BATCH_SIZE,
#     shuffle=False,
#     num_workers=0,
#     drop_last=False,
# )


# dataset_local_inference = Dataset_RNN(X_val, mode='test')
# local_inference_loader = torch.utils.data.DataLoader(
#     dataset_local_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

# dataset_inference = Dataset_RNN(df_val, mode='test')
# inference_loader = torch.utils.data.DataLoader(
#     dataset_inference, batch_size=INFER_BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False,
# )

In [89]:
# %%time

# X_train["train"] = X_train["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_train["val"] = X_train["val"].progress_apply(lambda x: torch.tensor(x).float())
# X_val["train"] = X_val["train"].progress_apply(lambda x: torch.tensor(x).long())
# X_val["val"] = X_val["val"].progress_apply(lambda x: torch.tensor(x).float())

In [90]:
%%time
X_train = X_train.to_pandas()
X_val = X_val.to_pandas()

X_train_to_dataset = pd.DataFrame(X_train[config.col_train_seq].tolist(), index=X_train.index)
X_train_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_train_to_dataset.columns]
X_train_to_dataset[config.col_target] = X_train[config.col_target]

X_val_to_dataset = pd.DataFrame(X_val[config.col_train_seq].tolist(), index=X_val.index)
X_val_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_val_to_dataset.columns]
X_val_to_dataset[config.col_target] = X_val[config.col_target]

X_val_target_to_dataset = pd.DataFrame(X_val_to_dataset[config.col_target].tolist(), index=X_val_to_dataset.index)
X_val_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_val_target_to_dataset.columns]
X_val_to_dataset[X_val_target_to_dataset.columns] = X_val_target_to_dataset[X_val_target_to_dataset.columns]
X_val_to_dataset.drop([config.col_target], axis=1, inplace=True)

X_train = pl.from_pandas(X_train)
X_val = pl.from_pandas(X_val)

cols_pos = [col for col in X_train_to_dataset.columns if "pos" in col]
cols_target_val = [col for col in X_val_to_dataset.columns if "target" in col]

CPU times: user 2min 12s, sys: 31.5 s, total: 2min 43s
Wall time: 2min 43s


In [91]:
cols_target_val

['target_0', 'target_1', 'target_2']

In [92]:
X_train_to_dataset.head()

,pos_0,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,target
0,831662,433725.0,34008.0,433725.0,433725.0,433725.0,34008.0,34008.0,433725.0,34008.0,621802
1,671720,654957.0,831662.0,433725.0,34008.0,433725.0,433725.0,433725.0,34008.0,34008.0,433725
2,901386,831662.0,654957.0,671720.0,654957.0,831662.0,NaN,NaN,NaN,NaN,433725
3,901386,831662.0,654957.0,671720.0,654957.0,831662.0,433725.0,34008.0,433725.0,NaN,433725
4,831662,654957.0,671720.0,654957.0,831662.0,433725.0,34008.0,433725.0,433725.0,433725.0,34008


In [93]:
X_val_to_dataset.head()

,pos_0,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,target_0,target_1,target_2
0,63656,404040.0,713773.0,647060.0,NaN,NaN,NaN,NaN,NaN,NaN,1257293,1257293,1257293
1,799059,954978.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,644778,1674216,644778
2,948806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,270070,943295,943295
3,29121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,669989,14530,1006198
4,633595,624605.0,633595.0,633595.0,633595.0,176268.0,633595.0,365077.0,NaN,NaN,356824,356824,356824


In [94]:
%%time

X_train_to_dataset.to_parquet('train.parquet', index=False)
X_val_to_dataset.to_parquet('valid.parquet', index=False)

CPU times: user 13.5 s, sys: 1.08 s, total: 14.5 s
Wall time: 13.5 s


In [95]:
%%time

train_ds = Dataset_merlin('train.parquet')
train_dl_merlin = Loader(dataset=train_ds, batch_size=config.batch_size_train, shuffle=True)

valid_ds = Dataset_merlin('valid.parquet')
valid_dl_merlin = Loader(dataset=valid_ds, batch_size=config.batch_size_infer, shuffle=False)

CPU times: user 8 s, sys: 8.95 s, total: 16.9 s
Wall time: 2.63 s


In [96]:
import math


def get_matrix_and_target_from_batch(
    batch, cols_pos=cols_pos, cols_target_val=cols_target_val, pad=pad, train=True
):
    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad
    )

    if train:
        target = batch[config.col_target]
    else:
        target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
        for i in range(len(cols_target_val)):
            target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    return batch_matrix, target, lens

In [97]:
del df, X_train, X_train_to_dataset, X_val_to_dataset
gc.collect()

4105

In [98]:
%%time

for batch, _ in tqdm(train_dl_merlin):
    # batch_matrix, batch["val"], lens = get_matrix_and_target_from_batch(batch, cols_pos=cols_pos, pad=pad)
    continue
    # break
# 5 min

  0%|          | 0/89416 [00:00<?, ?it/s]

CPU times: user 32 s, sys: 26.2 s, total: 58.2 s
Wall time: 46.2 s


In [99]:
# %%time

# batch_matrix = np.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
# for i in range(len(cols_pos)):
#     batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
# batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not np.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]

In [100]:
# def get_matrix_and_target_from_batch(batch, cols_pos=cols_pos):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     # batch_target = torch.reshape(batch["val"], (1, -1))
#     return batch_matrix, batch["val"]

In [118]:
DEBUG = False


def do_inverse_transform(array, ordinal_encoder=ordinal_encoder):
    result = ordinal_encoder.inverse_transform(array.reshape(-1, 1)).reshape(1, -1)[0]
    result = [val for val in result if val is not None]
    return result


def flatten_list(l):
    return [item for sublist in l for item in sublist]


def valid_func(valid_loader, model, criterion):
    model.eval()

    bar = tqdm(valid_loader)
    PREDS = list()
    TARGETS = list()

    losses = list()
    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, targets, lens = get_matrix_and_target_from_batch(batch, train=False)
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            # category = category.to(device).long()
            # subcategory = subcategory.to(device).long()
            # region = region.to(device).long()
            # targets = targets.to(device).long()
            # targets = targets.squeeze(1)

            logits = model(aids, lens)
            # loss = criterion(logits, targets)
            # losses.append(loss.item())

            # PREDS += [torch.argmax(logits, 1).detach().cpu()]
            k = 20
            preds_cur = torch.topk(logits, k).indices.detach().cpu().numpy()
            # print(preds_cur.shape)
            # print(preds_cur)
            # preds_cur = [np.argpartition(-pred, k)[:k] for pred in preds_cur]
            # preds_cur = [
            #     do_inverse_transform(pred, ordinal_encoder=ordinal_encoder)
            #     for pred in preds_cur
            # ]
            PREDS.append(preds_cur)
            targets = [target.tolist() for target in targets]
            TARGETS.append(targets)

            # bar.set_description(f"loss: {loss.item():.5f}")

    # loss_valid = np.mean(losses)
    # print("loss_valid:", loss_valid)

    # PREDS = torch.cat(PREDS).cpu().tolist()
    PREDS = flatten_list(PREDS)
    TARGETS = flatten_list(TARGETS)
    # TARGETS = torch.cat(TARGETS).cpu().tolist()
    # print(len(PREDS), len(TARGETS))
    # print(TARGETS)
    # print(PREDS)
    df_val = pd.DataFrame({"idx": np.arange(len(TARGETS)), "preds": PREDS})
    df_val = df_val.explode("preds")

    df_val["preds"] = ordinal_encoder.inverse_transform(
        df_val["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_val = df_val.dropna()
    df_val["preds"] = df_val["preds"].astype(int)
    df_val = pl.from_pandas(df_val)
    df_val = (
        (
            df_val.groupby(["idx"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("idx")
        .to_pandas()
    )

    PREDS = df_val["preds"].tolist()
    PREDS = [list(pred) for pred in PREDS]
    df_val["targets"] = TARGETS

    metric_total = utils.recall_several_targets(PREDS, TARGETS)
    print("metric_total:", metric_total)

    return df_val


# def inference_func(inference_loader, model):
#     model.eval()

#     bar = tqdm(inference_loader)
#     preds = list()

#     with torch.no_grad():
#         for batch_idx, (descs, category, subcategory, region) in enumerate(bar):
#             descs = descs.to(device).long()
#             category = category.to(device).long()
#             subcategory = subcategory.to(device).long()
#             region = region.to(device).long()

#             logits = model(descs, category, subcategory, region)

#             preds += [logits.detach().cpu()]

#     preds = torch.cat(preds).cpu().numpy()
#     preds = 1.0 / (1.0 + np.exp(-preds))
#     preds = np.concatenate(preds, axis=0)
#     return preds


def train_func(
    train_loader,
    model,
    criterion,
    opts: list,
    amp=False,
    num_batches_to_check=None,
    num_batches_to_validate=None,
    valid_loader=None,
):
    model.train()
    bar = tqdm(train_loader)
    losses = list()
    if amp:
        scaler = torch.cuda.amp.GradScaler()

    start_time = time.perf_counter()
    batches_passed = 0
    for batch_idx, (batch, _) in enumerate(bar):
        if num_batches_to_check:
            if batch_idx % num_batches_to_check == 0 and batch_idx > 0:
                batches_passed += num_batches_to_check
                end_time = time.perf_counter()
                total_time = end_time - start_time
                print(f"batches passed: {batches_passed}")
                print(
                    f"last {num_batches_to_check} batches took {total_time / 60 :.4f} minutes"
                )
                print()
                start_time = time.perf_counter()

        aids, targets, lens = get_matrix_and_target_from_batch(batch)
        aids = aids.to(device).long()
        lens = lens.to(device).long()
        # category = category.to(device).long()
        # subcategory = subcategory.to(device).long()
        # region = region.to(device).long()

        targets = targets.to(device).long()
        targets = targets.squeeze(1)

        if DEBUG and batch_idx == 2:
            print("Debug Mode. Only train on first 1 batches.")
            break

        if amp:
            with torch.cuda.amp.autocast():
                logits = model(aids, lens)
                loss = criterion(logits)
                scaler.scale(loss).backward()  # loss.backward()
                for optimizer in opts:
                    scaler.step(optimizer)  # optimizer.step()
                    scaler.update()
        else:
            logits = model(aids, lens)
            # print(logits.size())
            loss = criterion(logits, targets)
            loss.backward()
        for optimizer in opts:
            optimizer.step()

        for optimizer in opts:
            optimizer.zero_grad()

        losses.append(loss.item())

        if num_batches_to_validate:
            if batch_idx % num_batches_to_validate == 0 and batch_idx > 0:
                print("batch_idx:", batch_idx)
                model.eval()
                preds_local = valid_func(valid_loader, model, criterion)
                model.train()
                print("_" * 10)
                print()
        # smooth_loss = np.mean(losses[-30:])

        # bar.set_description(f"smooth_loss: {smooth_loss:.5f}")

    loss_train = np.mean(losses)
    print("loss_train:", loss_train)
    return loss_train

In [102]:
out_features_num = pad  # + 1
out_features_num

954978

In [103]:
model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{2 + 2}_larger_lr_and_bs.pth",
    )
)

model.to(device)

/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.


/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.




RRNN(
  (embeddings): Embedding(954979, 100)
  (lstm_desc): GRU(100, 128, batch_first=True, dropout=0.1)
  (attention_descs): Attention()
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.1, inplace=True)
  (fc1): Linear(in_features=128, out_features=100, bias=True)
  (dropout2): Dropout(p=0.1, inplace=True)
  (fc2): Linear(in_features=100, out_features=954979, bias=True)
)

In [104]:
params_sparsed = list()
params_not_sparsed = list()
cnt = 0
for param in model.parameters():
    if cnt == 0:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)
    cnt += 1

In [105]:
params_sparsed[0].shape

torch.Size([954979, 100])

In [106]:
class MultipleOptimizer:
    def __init__(self, *optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for optimizer in self.optimizers:
            optimizer.zero_grad()

    def step(self):
        for optimizer in self.optimizers:
            optimizer.step()


class MultipleScheduler:
    def __init__(self, *schedulers):
        self.schedulers = schedulers

    def step(self):
        for scheduler in self.schedulers:
            scheduler.step()

In [107]:
init_lr = config.init_lr

criterion = nn.CrossEntropyLoss(ignore_index=pad)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
optimizer_sparsed = optim.SparseAdam(params_sparsed, lr=init_lr)
optimizer_not_sparsed = optim.Adam(params_not_sparsed, lr=init_lr)
opts = [optimizer]  # , optimizer_sparsed, optimizer_not_sparsed
opt = MultipleOptimizer(optimizer_sparsed, optimizer_not_sparsed)

# scheduler_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler_not_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler = MultipleScheduler(scheduler_sparsed, scheduler_not_sparsed)

seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [108]:
%%time

for epoch in tqdm(range(config.n_epochs), desc="epochs"):

    print(f"Epoch {epoch + 1} started")
    torch.cuda.empty_cache()

    # print("LR:", scheduler.schedulers[0].state_dict()["_last_lr"][0])
    loss_train = train_func(train_dl_merlin, model, criterion, opts, num_batches_to_validate=10_000, valid_loader=valid_dl_merlin)
    # scheduler.step()

    df_val = valid_func(valid_dl_merlin, model, criterion)
    # X_val['preds'] = preds_local
    # aucs = list()
    # for category, cur_df in X_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('local mean_auc:', np.mean(aucs))

    # loss_valid, preds = valid_func(valid_loader, model, criterion)
    # df_val['preds'] = preds
    # aucs = list()
    # for category, cur_df in df_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('mean_auc:', np.mean(aucs))

    torch.save(
        model.state_dict(),
        f"{config.model_name}_epoch{2 + 2 + epoch + 1}_larger_lr_and_bs.pth",
    )

    print("Checkpoint saved!")
    print()
    print("*" * 90)
    print()
    print()

# del model
torch.cuda.empty_cache()
gc.collect()

epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


  0%|          | 0/89416 [00:00<?, ?it/s]

batch_idx: 10000


  0%|          | 0/912 [00:00<?, ?it/s]

metric_total: 0.22343133579678323
batch_idx: 20000


... (34 outputs left out)

  0%|          | 0/912 [00:00<?, ?it/s]

metric_total: 0.2835409414662882
Checkpoint saved!

******************************************************************************************


Epoch 3 started


  0%|          | 0/89416 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [119]:
df_val = valid_func(valid_dl_merlin, model, criterion)

  0%|          | 0/912 [00:00<?, ?it/s]

metric_total: 0.2835409414662882


In [120]:
df_val

,idx,preds,targets
0,0,"[1257293, 801774, 1534690, 1493790, 372636, 11...","[1257293, 1257293, 1257293]"
1,1,"[1552386, 1464505, 1434122, 1636724, 337471, 1...","[644778, 1674216, 644778]"
2,2,"[1843528, 1596692, 1350258, 337471, 208769, 28...","[270070, 943295, 943295]"
3,3,"[56279, 923948, 1609228, 502010, 1121939, 7772...","[669989, 14530, 1006198]"
4,4,"[709657, 1231260, 822475, 1383779, 816055, 886...","[356824, 356824, 356824]"
...,...,...,...
466820,466820,"[1027325, 702989, 1742471, 577378, 1835418, 34...","[1027325, 1027325, 1027325]"
466821,466821,"[508776, 185306, 1182614, 1586171, 1787028, 87...","[1643173, 234336, 916301]"
466822,466822,"[1269599, 1686700, 105428, 1457195, 588069, 11...","[1686700, 1129507, 1686700]"
466823,466823,"[1124813, 1338994, 247477, 1065138, 887206, 13...","[728208, 1179279, 828731]"


In [ ]:
val["len_seq"] = val[config.col_train_seq].apply(len)

In [109]:
# metric_total: 0.042083333333333334 1 ep
# metric_total: 0.06625 1 ep 5 iter
# metric_total: 0.059583333333333335 1 ep 10 iter
# metric_total: 0.0725 1 ep 15 iter

In [110]:
X_val[config.col_target].value_counts().head(10)

target,counts
list[i64],u32
"[1299872, 1615067, 245870]",1
"[700090, 781749, 1718934]",1
"[111593, 398618, 111593]",1
"[733383, 247184, 229788]",1
"[775851, 128912, 1127094]",1
"[675652, 675652, 1423853]",1
"[1196254, 770777, 1342487]",1
"[218753, 75492, 1755218]",1
"[60810, 982728, 1021295]",1


In [111]:
X_train["val"].value_counts().head(10)

NameError: name 'X_train' is not defined

In [ ]:
X_val = X_val.to_pandas()

In [ ]:
X_val["preds"] = preds_local
X_val["recall_row"] = X_val.progress_apply(
    lambda x: utils.recall_several_targets_one_row(
        x["preds"], x[config.col_target].tolist()
    ),
    axis=1,
)

In [ ]:
X_val.shape

In [ ]:
X_val.tail()

In [ ]:
stop

In [ ]:
X_val[X_val["recall_row"] == 1].shape

In [ ]:
X_val_true = X_val[X_val["recall_row"] == 1]

In [ ]:
pad

In [ ]:
X_val_true.head()

In [ ]:
X_train[X_train["val"] == pad].shape[0] / X_train.shape[0]

In [ ]:
X_val_true.iloc[0]["preds"]

In [ ]:
stop
# loss_train: 12.190348625183105

# loss_train: 9.231197555859884
# loss_valid: 20.559587955474854
# metric_total: 0.000999000999000999

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
for param in model.parameters():
    if param.grad.is_sparse:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)

In [ ]:
params_sparsed

In [ ]:
df_train.groupby(["n_week"]).count().sort("n_week", reverse=False)

In [ ]:
df_train_ex = df_train.filter(pl.col("user") <= 10000)

In [ ]:
%%time

dict_train = dict()
dict_answers = dict()
dict_val_len = dict()
dict_user = dict()
key = 1

for df_user in tqdm(df_train.groupby(["user"], maintain_order=True)):
    df_weeks_count = df_user.groupby(["n_week"]).count().sort("n_week", reverse=False)
    if len(df_weeks_count) <= 1:
        continue
    user = df_user["user"][0]
    for i in range(len(df_weeks_count["n_week"]) - 1):
        n_week_cur = df_weeks_count["n_week"][i]
        val_len = 34 - n_week_cur
        dict_train[key] = df_user.filter(pl.col("n_week") == n_week_cur)
        dict_answers[key] = df_user.filter(pl.col("n_week") > n_week_next)
        dict_val_len[key] = val_len
        dict_user[key] = user
        key += 1

In [ ]:
dict_train[1]

In [ ]:
dict_answers[1]

In [ ]:
df_user.filter(pl.col("n_week") == n_week_cur)

In [ ]:
for i in range(len(df_weeks_count["n_week"]) - 1):
    n_week_cur = df_weeks_count["n_week"][i]
    n_week_next = df_weeks_count["n_week"][i + 1]
    print(n_week_cur, n_week_next)

In [ ]:
df_user

In [ ]:
df_train_ex

In [ ]:
df_train.tail(10)

In [ ]:
df_test.head()

In [ ]:
utils.plot_user_activity(user=12899774, df=df_train)

In [ ]:
df_train["type"].value_counts()

In [ ]:
df_train["date"].min(), df_train["date"].max()

In [ ]:
df_test["date"].min(), df_test["date"].max()

In [ ]:
%%time

df_train.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
%%time

df_test.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
df_train_n_session = (df_train.groupby("user")[["n_session"]].max() + 1).reset_index()
df_test_n_session = (df_test.groupby("user")[["n_session"]].max() + 1).reset_index()

In [ ]:
df_train_n_session["n_session"].value_counts()

In [ ]:
df_test_n_session["n_session"].value_counts()

In [ ]:
def plot_n_interactions_per_dayofweek(df):
    try:
        df_name = df.name
    except AttributeError:
        df_name = None

    title = "n_interactions per dayofweek"
    if df_name is not None:
        title += f" in {df_name}"

    plt.figure(figsize=(12, 7))
    plt.title(title)
    plt.xlabel("dayofweek")
    plt.ylabel("n_interactions")

    dict_type_inter_offset = dict()
    dict_type_inter_offset["clicks"] = -0.2
    dict_type_inter_offset["carts"] = 0.0
    dict_type_inter_offset["orders"] = 0.2

    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "g"
    dict_type_inter_color["orders"] = "r"

    for (dayofweek, type_inter), df_cur in df.groupby(["dayofweek", "type"]):
        ys = df_cur["ts"].count()
        plt.bar(
            dayofweek + dict_type_inter_offset[type_inter],
            ys,
            width=0.2,
            color=dict_type_inter_color[type_inter],
            align="center",
            # label=feature,
            # marker=",",
            # linestyle="dashed",
            alpha=0.6,
        )

In [ ]:
plot_n_interactions_per_dayofweek(df_train)

In [ ]:
plot_n_interactions_per_dayofweek(df_test)

In [ ]:
def plot_user_activity(user, df):
    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "orange"
    dict_type_inter_color["orders"] = "r"

    offset = 0.5
    df_cur = df[df["user"] == user]
    df_cur["date_day"] = df_cur["date_day"].astype(str)

    if df.name == "df_train":
        x = pd.date_range("2022-08-01", "2022-08-28")
    elif df.name == "df_test":
        x = pd.date_range("2022-08-29", "2022-09-04")
    x = [str(d).split(" ")[0] for d in x]
    map_dates = {x[i]: i for i in range(len(x))}
    points_sundays = [point for point in range(len(x)) if (point + 1) % 7 == 0]

    plt.figure(figsize=(12, 7))
    plt.xticks(range(len(x)), x, rotation=90)
    plt.yticks(range(25))

    for point in range(25):
        plt.axhline(y=point, color="gray", linestyle="-", alpha=0.1)
    for point in range(len(x)):
        plt.axvline(x=point + offset, color="gray", linestyle="-", alpha=0.1)
        plt.axvline(x=point - offset, color="gray", linestyle="-", alpha=0.1)
    for point in points_sundays:
        plt.axvline(x=point + offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset - 1, color="r", linestyle="-", alpha=0.3)

    for action, df_cur_action in df_cur.groupby("type"):
        xx = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        yy = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        plt.scatter(
            df_cur_action["date_day"].map(map_dates) + xx,
            df_cur_action["hour"] + yy,
            c=dict_type_inter_color[action],
            s=10,
            label=action,
        )
    plt.legend(
        loc="upper right", bbox_to_anchor=(1.1, 1.0), ncol=1, fancybox=True, shadow=True
    )

In [ ]:
plot_user_activity(user=2932269, df=df_train)

In [ ]:
plot_user_activity(user=12899780, df=df_test)